# Water Potability: Classical Model Comparison with scikit-learn

## Project Overview

This notebook compares k-nearest neighbors, two Naive Bayes variants, decision trees, and majority-vote ensembles for predicting whether a water sample is potable.

The analysis evaluates multiple train/test splits, normalized and unnormalized representations, cross-validation strategies, model-specific hyperparameter searches, and ensemble behavior.

### Technical Coverage

- k-nearest neighbors
- Multinomial Naive Bayes
- Bernoulli Naive Bayes
- decision trees
- three holdout configurations
- row normalization
- 10-fold cross-validation
- leave-one-out cross-validation
- hyperparameter search
- majority-vote ensembles
- baseline vs. tuned model comparison

## 1. Dataset and Prediction Target

The dataset contains **2,011 complete water samples**, nine chemical or physical measurements, and a binary `Potability` target.

| Feature | Description |
|---|---|
| `ph` | acidity or alkalinity |
| `Hardness` | dissolved calcium- and magnesium-related hardness |
| `Solids` | total dissolved solids |
| `Chloramines` | disinfectant concentration |
| `Sulfate` | sulfate concentration |
| `Conductivity` | electrical conductivity |
| `Organic_carbon` | total organic carbon |
| `Trihalomethanes` | trihalomethane concentration |
| `Turbidity` | suspended-matter indicator |
| `Potability` | 1 for potable water, 0 otherwise |

## 2. Environment

In [31]:
import itertools
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, LeaveOneOut
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.preprocessing import Normalizer


## 3. Data Loading and Class Balance

In [32]:
from google.colab import files

uploaded = files.upload()

Saving water_potability.xlsx to water_potability (1).xlsx


In [33]:
import pandas as pd
import numpy as np

water_df = pd.read_excel("water_potability.xlsx")

water_np = water_df.to_numpy()

print("Dataset shape:", water_np.shape)

print(
    "Class counts [not potable, potable]:",
    np.bincount(water_np[:, 9].astype(int))
)

print("Feature matrix shape:", water_np[:, 0:9].shape)

water_df.head()

Dataset shape: (2011, 10)
Class counts [not potable, potable]: [1200  811]
Feature matrix shape: (2011, 9)


,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
0,7.899452,210.734124,15896.365940,6.907203,319.886957,448.666423,18.169921,124.000000,2.853767,1
1,6.145148,197.541072,39657.272110,9.900159,288.157883,319.434033,11.587378,120.030077,4.600886,0
2,5.036454,190.164520,29258.738140,4.991061,300.475925,332.359715,11.055801,116.161622,3.534665,1
3,8.969697,195.744765,9049.682595,7.467068,396.453568,378.528511,17.757697,114.208671,3.983099,0
4,8.285072,151.573778,14402.726700,9.050080,303.081838,322.521815,13.652653,114.034946,4.274661,1


## 4. Baseline Models on Raw Features

Four baseline classifiers are evaluated using 20%, 25%, and 30% test partitions. The same split seeds are retained across the project so model comparisons remain consistent.

### 20% Test Split

In [34]:
x_train, x_test, y_train, y_test = train_test_split(water_np[:, 0:9], water_np[:,9].astype(int), test_size=0.20, random_state=8684)

model =  KNeighborsClassifier()
model.fit(x_train, y_train)
pred = model.predict(x_test)
print('Accuracy for KNeighborsClassifier = {:.4f}'.format(accuracy_score(pred,y_test)))

model =  MultinomialNB()
model.fit(x_train, y_train)
pred = model.predict(x_test)
print('Accuracy for MultinomialNB = {:.4f}'.format(accuracy_score(pred,y_test)))

model =  BernoulliNB()
model.fit(x_train, y_train)
pred = model.predict(x_test)
print('Accuracy for BernoulliNB = {:.4f}'.format(accuracy_score(pred,y_test)))

model =  DecisionTreeClassifier()
model.fit(x_train, y_train)
pred = model.predict(x_test)
print('Accuracy for DecisionTreeClassifier = {:.4f}'.format(accuracy_score(pred,y_test)))

Accuracy for KNeighborsClassifier = 0.5881
Accuracy for MultinomialNB = 0.5434
Accuracy for BernoulliNB = 0.6700
Accuracy for DecisionTreeClassifier = 0.6203


### 25% Test Split

In [35]:
x_train, x_test, y_train, y_test = train_test_split(water_np[:, 0:9], water_np[:,9].astype(int), test_size=0.25, random_state=7557)

model =  KNeighborsClassifier()
model.fit(x_train, y_train)
pred = model.predict(x_test)
print('Accuracy for KNeighborsClassifier = {:.4f}'.format(accuracy_score(pred,y_test)))

model =  MultinomialNB()
model.fit(x_train, y_train)
pred = model.predict(x_test)
print('Accuracy for MultinomialNB = {:.4f}'.format(accuracy_score(pred,y_test)))

model =  BernoulliNB()
model.fit(x_train, y_train)
pred = model.predict(x_test)
print('Accuracy for BernoulliNB = {:.4f}'.format(accuracy_score(pred,y_test)))

model =  DecisionTreeClassifier()
model.fit(x_train, y_train)
pred = model.predict(x_test)
print('Accuracy for DecisionTreeClassifier = {:.4f}'.format(accuracy_score(pred,y_test)))

Accuracy for KNeighborsClassifier = 0.5865
Accuracy for MultinomialNB = 0.5408
Accuracy for BernoulliNB = 0.6322
Accuracy for DecisionTreeClassifier = 0.6163


### 30% Test Split

In [36]:
x_train, x_test, y_train, y_test = train_test_split(water_np[:, 0:9], water_np[:,9].astype(int), test_size=0.30, random_state=4907)

model =  KNeighborsClassifier()
model.fit(x_train, y_train)
pred = model.predict(x_test)
print('Accuracy for KNeighborsClassifier = {:.4f}'.format(accuracy_score(pred,y_test)))

model =  MultinomialNB()
model.fit(x_train, y_train)
pred = model.predict(x_test)
print('Accuracy for MultinomialNB = {:.4f}'.format(accuracy_score(pred,y_test)))

model =  BernoulliNB()
model.fit(x_train, y_train)
pred = model.predict(x_test)
print('Accuracy for BernoulliNB = {:.4f}'.format(accuracy_score(pred,y_test)))

model =  DecisionTreeClassifier()
model.fit(x_train, y_train)
pred = model.predict(x_test)
print('Accuracy for DecisionTreeClassifier = {:.4f}'.format(accuracy_score(pred,y_test)))

Accuracy for KNeighborsClassifier = 0.5960
Accuracy for MultinomialNB = 0.5613
Accuracy for BernoulliNB = 0.6391
Accuracy for DecisionTreeClassifier = 0.6076


## 5. Row-Normalized Feature Representation

The nine input attributes are normalized before refitting the same four baseline classifiers.

In [37]:
y = water_np[:,9].astype(int)
transformer = Normalizer().fit(water_np[:, 0:9])
water_np_normalized = transformer.transform(water_np[:, 0:9])

### 20% Test Split

In [38]:
x_train, x_test, y_train, y_test = train_test_split(water_np_normalized, y, test_size=0.20, random_state=8684)

model =  KNeighborsClassifier()
model.fit(x_train, y_train)
pred = model.predict(x_test)
print('Accuracy for KNeighborsClassifier = {:.4f}'.format(accuracy_score(pred,y_test)))

model =  MultinomialNB()
model.fit(x_train, y_train)
pred = model.predict(x_test)
print('Accuracy for MultinomialNB = {:.4f}'.format(accuracy_score(pred,y_test)))

model =  BernoulliNB()
model.fit(x_train, y_train)
pred = model.predict(x_test)
print('Accuracy for BernoulliNB = {:.4f}'.format(accuracy_score(pred,y_test)))

model =  DecisionTreeClassifier()
model.fit(x_train, y_train)
pred = model.predict(x_test)
print('Accuracy for DecisionTreeClassifier = {:.4f}'.format(accuracy_score(pred,y_test)))

Accuracy for KNeighborsClassifier = 0.5633
Accuracy for MultinomialNB = 0.6700
Accuracy for BernoulliNB = 0.6700
Accuracy for DecisionTreeClassifier = 0.5856


### 25% Test Split

In [39]:
x_train, x_test, y_train, y_test = train_test_split(water_np_normalized, y, test_size=0.25, random_state=7557)

model =  KNeighborsClassifier()
model.fit(x_train, y_train)
pred = model.predict(x_test)
print('Accuracy for KNeighborsClassifier = {:.4f}'.format(accuracy_score(pred,y_test)))

model =  MultinomialNB()
model.fit(x_train, y_train)
pred = model.predict(x_test)
print('Accuracy for MultinomialNB = {:.4f}'.format(accuracy_score(pred,y_test)))

model =  BernoulliNB()
model.fit(x_train, y_train)
pred = model.predict(x_test)
print('Accuracy for BernoulliNB = {:.4f}'.format(accuracy_score(pred,y_test)))

model =  DecisionTreeClassifier()
model.fit(x_train, y_train)
pred = model.predict(x_test)
print('Accuracy for DecisionTreeClassifier = {:.4f}'.format(accuracy_score(pred,y_test)))

Accuracy for KNeighborsClassifier = 0.5626
Accuracy for MultinomialNB = 0.6322
Accuracy for BernoulliNB = 0.6322
Accuracy for DecisionTreeClassifier = 0.5686


### 30% Test Split

In [40]:
x_train, x_test, y_train, y_test = train_test_split(water_np_normalized, y, test_size=0.30, random_state=4907)

model =  KNeighborsClassifier()
model.fit(x_train, y_train)
pred = model.predict(x_test)
print('Accuracy for KNeighborsClassifier = {:.4f}'.format(accuracy_score(pred,y_test)))

model =  MultinomialNB()
model.fit(x_train, y_train)
pred = model.predict(x_test)
print('Accuracy for MultinomialNB = {:.4f}'.format(accuracy_score(pred,y_test)))

model =  BernoulliNB()
model.fit(x_train, y_train)
pred = model.predict(x_test)
print('Accuracy for BernoulliNB = {:.4f}'.format(accuracy_score(pred,y_test)))

model =  DecisionTreeClassifier()
model.fit(x_train, y_train)
pred = model.predict(x_test)
print('Accuracy for DecisionTreeClassifier = {:.4f}'.format(accuracy_score(pred,y_test)))

Accuracy for KNeighborsClassifier = 0.5778
Accuracy for MultinomialNB = 0.6391
Accuracy for BernoulliNB = 0.6391
Accuracy for DecisionTreeClassifier = 0.5430


## 6. Cross-Validation

Because the complete-data dataset is relatively small, the baseline models are also evaluated with 10-fold cross-validation and leave-one-out cross-validation.

### 10-Fold Cross-Validation

In [41]:
y = water_np[:,9].astype(int)
X = water_np[:, 0:9]
kf = KFold(n_splits=10)
accuracy = 0.0
for train, test in kf.split(X):
  model =  KNeighborsClassifier()
  model.fit(X[train], y[train])
  pred = model.predict(X[test])
  accuracy += accuracy_score(pred,y[test])
print('Accuracy for KNeighborsClassifier = {:.4f}'.format(accuracy/10))

Accuracy for KNeighborsClassifier = 0.5520


In [42]:
y = water_np[:,9].astype(int)
X = water_np[:, 0:9]
kf = KFold(n_splits=10)
accuracy = 0.0
for train, test in kf.split(X):
  model =  MultinomialNB()
  model.fit(X[train], y[train])
  pred = model.predict(X[test])
  accuracy += accuracy_score(pred,y[test])
print('Accuracy for MultinomialNB = {:.4f}'.format(accuracy/10))

Accuracy for MultinomialNB = 0.5216


In [43]:
y = water_np[:,9].astype(int)
X = water_np[:, 0:9]
kf = KFold(n_splits=10)
accuracy = 0.0
for train, test in kf.split(X):
  model =  BernoulliNB()
  model.fit(X[train], y[train])
  pred = model.predict(X[test])
  accuracy += accuracy_score(pred,y[test])
print('Accuracy for BernoulliNB = {:.4f}'.format(accuracy/10))

Accuracy for BernoulliNB = 0.5967


In [44]:
y = water_np[:,9].astype(int)
X = water_np[:, 0:9]
kf = KFold(n_splits=10)
accuracy = 0.0
for train, test in kf.split(X):
  model =  DecisionTreeClassifier()
  model.fit(X[train], y[train])
  pred = model.predict(X[test])
  accuracy += accuracy_score(pred,y[test])
print('Accuracy for DecisionTreeClassifier = {:.4f}'.format(accuracy/10))

Accuracy for DecisionTreeClassifier = 0.5997


### Leave-One-Out Cross-Validation

In [45]:
y = water_np[:,9].astype(int)
X = water_np[:, 0:9]
loo = LeaveOneOut()
accuracy = 0.0
for train, test in loo.split(X):
  model =  KNeighborsClassifier()
  model.fit(X[train], y[train])
  pred = model.predict(X[test])
  accuracy += accuracy_score(pred,y[test])
print('Accuracy for KNeighborsClassifier = {:.4f}'.format(accuracy/X.shape[0]))

Accuracy for KNeighborsClassifier = 0.5485


In [46]:
y = water_np[:,9].astype(int)
X = water_np[:, 0:9]
loo = LeaveOneOut()
accuracy = 0.0
for train, test in loo.split(X):
  model =  MultinomialNB()
  model.fit(X[train], y[train])
  pred = model.predict(X[test])
  accuracy += accuracy_score(pred,y[test])
print('Accuracy for MultinomialNB = {:.4f}'.format(accuracy/X.shape[0]))

Accuracy for MultinomialNB = 0.5236


In [47]:
y = water_np[:,9].astype(int)
X = water_np[:, 0:9]
loo = LeaveOneOut()
accuracy = 0.0
for train, test in loo.split(X):
  model =  BernoulliNB()
  model.fit(X[train], y[train])
  pred = model.predict(X[test])
  accuracy += accuracy_score(pred,y[test])
print('Accuracy for BernoulliNB = {:.4f}'.format(accuracy/X.shape[0]))

Accuracy for BernoulliNB = 0.5967


In [48]:
y = water_np[:,9].astype(int)
X = water_np[:, 0:9]
loo = LeaveOneOut()
accuracy = 0.0
for train, test in loo.split(X):
  model =  DecisionTreeClassifier()
  model.fit(X[train], y[train])
  pred = model.predict(X[test])
  accuracy += accuracy_score(pred,y[test])
print('Accuracy for DecisionTreeClassifier = {:.4f}'.format(accuracy/X.shape[0]))

Accuracy for DecisionTreeClassifier = 0.5903


## 7. Majority-Vote Ensemble with Baseline Models

Predictions from k-nearest neighbors, MultinomialNB, BernoulliNB, and a decision tree are combined through majority voting.

### 20% Test Split

In [49]:
x_train, x_test, y_train, y_test = train_test_split(water_np[:, 0:9], water_np[:,9].astype(int), test_size=0.20, random_state=8684)
pred = np.zeros((4, y_test.shape[0]), dtype=int)

model =  KNeighborsClassifier()
model.fit(x_train, y_train)
pred[0,:] = model.predict(x_test)

model =  MultinomialNB()
model.fit(x_train, y_train)
pred[1,:] = model.predict(x_test)

model =  BernoulliNB()
model.fit(x_train, y_train)
pred[2,:] = model.predict(x_test)

model =  DecisionTreeClassifier()
model.fit(x_train, y_train)
pred[3,:] = model.predict(x_test)

pred = np.array([np.argmax(np.bincount(pred[:,i])) for i in range(y_test.shape[0])])

print('Accuracy for EnsembleClassifier = {:.4f}'.format(accuracy_score(pred,y_test)))

Accuracy for EnsembleClassifier = 0.6749


### 25% Test Split

In [50]:


x_train, x_test, y_train, y_test = train_test_split(water_np[:, 0:9], water_np[:,9].astype(int), test_size=0.25, random_state=7557)
pred = np.zeros((4, y_test.shape[0]), dtype=int)

model =  KNeighborsClassifier()
model.fit(x_train, y_train)
pred[0,:] = model.predict(x_test)

model =  MultinomialNB()
model.fit(x_train, y_train)
pred[1,:] = model.predict(x_test)

model =  BernoulliNB()
model.fit(x_train, y_train)
pred[2,:] = model.predict(x_test)

model =  DecisionTreeClassifier()
model.fit(x_train, y_train)
pred[3,:] = model.predict(x_test)

pred = np.array([np.argmax(np.bincount(pred[:,i])) for i in range(y_test.shape[0])])

print('Accuracy for EnsembleClassifier = {:.4f}'.format(accuracy_score(pred,y_test)))

Accuracy for EnsembleClassifier = 0.6561


### 30% Test Split

In [51]:


x_train, x_test, y_train, y_test = train_test_split(water_np[:, 0:9], water_np[:,9].astype(int), test_size=0.30, random_state=4907)
pred = np.zeros((4, y_test.shape[0]), dtype=int)

model =  KNeighborsClassifier()
model.fit(x_train, y_train)
pred[0,:] = model.predict(x_test)

model =  MultinomialNB()
model.fit(x_train, y_train)
pred[1,:] = model.predict(x_test)

model =  BernoulliNB()
model.fit(x_train, y_train)
pred[2,:] = model.predict(x_test)

model =  DecisionTreeClassifier()
model.fit(x_train, y_train)
pred[3,:] = model.predict(x_test)

pred = np.array([np.argmax(np.bincount(pred[:,i])) for i in range(y_test.shape[0])])

print('Accuracy for EnsembleClassifier = {:.4f}'.format(accuracy_score(pred,y_test)))

Accuracy for EnsembleClassifier = 0.6573


## 8. Hyperparameter Search

Each model family is tuned independently for the same three train/test partitions.

### k-Nearest Neighbors

#### 30% Test Split

In [52]:
# KNeighborsClassifier
n_neighbors = np.array([2,3,4,5,6,7,8,9])
weights = np.array(['uniform', 'distance'])
algorithms = np.array(['ball_tree', 'kd_tree', 'brute'])
ps = np.array([1,2,3])

x_train, x_test, y_train, y_test = train_test_split(water_np[:, 0:9], water_np[:,9].astype(int), test_size=0.30, random_state=4907)
accuracy = 0.0
for n_neighbor, weight, algorithm, p in itertools.product(n_neighbors, weights, algorithms, ps):
  model =  KNeighborsClassifier(n_neighbors=n_neighbor, weights=weight, algorithm=algorithm, p=p)
  model.fit(x_train, y_train)
  pred = model.predict(x_test)
  if accuracy < accuracy_score(pred,y_test):
    accuracy = accuracy_score(pred,y_test)
    print(n_neighbor, weight, algorithm, p, accuracy)

print('Accuracy for KNeighborsClassifier = {:.4f}'.format(accuracy))

2 uniform ball_tree 1 0.6307947019867549
Accuracy for KNeighborsClassifier = 0.6308


#### 25% Test Split

In [53]:
# KNeighborsClassifier
n_neighbors = np.array([2,3,4,5,6,7,8,9])
weights = np.array(['uniform', 'distance'])
algorithms = np.array(['ball_tree', 'kd_tree', 'brute'])
ps = np.array([1,2,3])

x_train, x_test, y_train, y_test = train_test_split(water_np[:, 0:9], water_np[:,9].astype(int), test_size=0.25, random_state=7557)
accuracy = 0.0
for n_neighbor, weight, algorithm, p in itertools.product(n_neighbors, weights, algorithms, ps):
  model =  KNeighborsClassifier(n_neighbors=n_neighbor, weights=weight, algorithm=algorithm, p=p)
  model.fit(x_train, y_train)
  pred = model.predict(x_test)
  if accuracy < accuracy_score(pred,y_test):
    accuracy = accuracy_score(pred,y_test)
    print(n_neighbor, weight, algorithm, p, accuracy)

print('Accuracy for KNeighborsClassifier = {:.4f}'.format(accuracy))

2 uniform ball_tree 1 0.610337972166998
2 uniform ball_tree 2 0.614314115308151
Accuracy for KNeighborsClassifier = 0.6143


#### 20% Test Split

In [54]:
# KNeighborsClassifier
n_neighbors = np.array([2,3,4,5,6,7,8,9])
weights = np.array(['uniform', 'distance'])
algorithms = np.array(['ball_tree', 'kd_tree', 'brute'])
ps = np.array([1,2,3])

x_train, x_test, y_train, y_test = train_test_split(water_np[:, 0:9], water_np[:,9].astype(int), test_size=0.20, random_state=8684)
accuracy = 0.0
for n_neighbor, weight, algorithm, p in itertools.product(n_neighbors, weights, algorithms, ps):
  model =  KNeighborsClassifier(n_neighbors=n_neighbor, weights=weight, algorithm=algorithm, p=p)
  model.fit(x_train, y_train)
  pred = model.predict(x_test)
  if accuracy < accuracy_score(pred,y_test):
    accuracy = accuracy_score(pred,y_test)
    print(n_neighbor, weight, algorithm, p, accuracy)

print('Accuracy for KNeighborsClassifier = {:.4f}'.format(accuracy))

2 uniform ball_tree 1 0.6004962779156328
2 uniform ball_tree 2 0.6178660049627791
Accuracy for KNeighborsClassifier = 0.6179


### Multinomial Naive Bayes

#### 20% Test Split

In [55]:
# MultinomialNB
alphas = np.array([1.0, 0.05, 0.1])
fit_priors = np.array([True, False])

x_train, x_test, y_train, y_test = train_test_split(water_np[:, 0:9], water_np[:,9].astype(int), test_size=0.20, random_state=8684)
accuracy = 0.0
for alpha, fit_prior in itertools.product(alphas, fit_priors):
  model =  MultinomialNB(alpha=alpha, fit_prior=fit_prior)
  model.fit(x_train, y_train)
  pred = model.predict(x_test)
  if accuracy < accuracy_score(pred,y_test):
    accuracy = accuracy_score(pred,y_test)
    print(alpha, fit_prior, accuracy)

print('Accuracy for MultinomialNB = {:.4f}'.format(accuracy))

1.0 True 0.543424317617866
Accuracy for MultinomialNB = 0.5434


#### 25% Test Split

In [56]:
# MultinomialNB
alphas = np.array([1.0, 0.05, 0.1])
fit_priors = np.array([True, False])

x_train, x_test, y_train, y_test = train_test_split(water_np[:, 0:9], water_np[:,9].astype(int), test_size=0.25, random_state=7557)
accuracy = 0.0
for alpha, fit_prior in itertools.product(alphas, fit_priors):
  model =  MultinomialNB(alpha=alpha, fit_prior=fit_prior)
  model.fit(x_train, y_train)
  pred = model.predict(x_test)
  if accuracy < accuracy_score(pred,y_test):
    accuracy = accuracy_score(pred,y_test)
    print(alpha, fit_prior, accuracy)

print('Accuracy for MultinomialNB = {:.4f}'.format(accuracy))

1.0 True 0.5407554671968191
1.0 False 0.5427435387673957
Accuracy for MultinomialNB = 0.5427


#### 30% Test Split

In [57]:
# MultinomialNB
alphas = np.array([1.0, 0.05, 0.1])
fit_priors = np.array([True, False])

x_train, x_test, y_train, y_test = train_test_split(water_np[:, 0:9], water_np[:,9].astype(int), test_size=0.30, random_state=4907)
accuracy = 0.0
for alpha, fit_prior in itertools.product(alphas, fit_priors):
  model =  MultinomialNB(alpha=alpha, fit_prior=fit_prior)
  model.fit(x_train, y_train)
  pred = model.predict(x_test)
  if accuracy < accuracy_score(pred,y_test):
    accuracy = accuracy_score(pred,y_test)
    print(alpha, fit_prior, accuracy)

print('Accuracy for MultinomialNB = {:.4f}'.format(accuracy))

1.0 True 0.5612582781456954
Accuracy for MultinomialNB = 0.5613


### Bernoulli Naive Bayes

#### 20% Test Split

In [58]:
# BernoulliNB
alphas = alphas = np.array([1.0, 0.05, 0.1])
binarizes = np.array([0.0, 0.5, 1.0, np.mean(x_train)])
fit_priors = np.array([True, False])

x_train, x_test, y_train, y_test = train_test_split(water_np[:, 0:9], water_np[:,9].astype(int), test_size=0.20, random_state=8684)
accuracy = 0.0
for alpha, binarize, fit_prior in itertools.product(alphas, binarizes, fit_priors):

  model =  BernoulliNB(alpha=alpha, binarize=binarize, fit_prior=fit_prior)
  model.fit(x_train, y_train)
  pred = model.predict(x_test)
  if accuracy < accuracy_score(pred,y_test):
    accuracy = accuracy_score(pred,y_test)
    print(alpha,binarize, fit_prior, accuracy)

print('Accuracy for BernoulliNB = {:.4f}'.format(accuracy))

1.0 0.0 True 0.6699751861042184
1.0 1.0 True 0.6724565756823822
Accuracy for BernoulliNB = 0.6725


#### 25% Test Split

In [59]:
# BernoulliNB
alphas = alphas = np.array([1.0, 0.05, 0.1])
binarizes = np.array([0.0, 0.5, 1.0, np.mean(x_train)])
fit_priors = np.array([True, False])

x_train, x_test, y_train, y_test = train_test_split(water_np[:, 0:9], water_np[:,9].astype(int), test_size=0.25, random_state=7557)
accuracy = 0.0
for alpha, binarize, fit_prior in itertools.product(alphas, binarizes, fit_priors):

  model =  BernoulliNB(alpha=alpha, binarize=binarize, fit_prior=fit_prior)
  model.fit(x_train, y_train)
  pred = model.predict(x_test)
  if accuracy < accuracy_score(pred,y_test):
    accuracy = accuracy_score(pred,y_test)
    print(alpha,binarize, fit_prior, accuracy)

print('Accuracy for BernoulliNB = {:.4f}'.format(accuracy))

1.0 0.0 True 0.6322067594433399
Accuracy for BernoulliNB = 0.6322


#### 30% Test Split

In [60]:
# BernoulliNB
alphas = alphas = np.array([1.0, 0.05, 0.1])
binarizes = np.array([0.0, 0.5, 1.0, np.mean(x_train)])
fit_priors = np.array([True, False])

x_train, x_test, y_train, y_test = train_test_split(water_np[:, 0:9], water_np[:,9].astype(int), test_size=0.30, random_state=4907)
accuracy = 0.0
for alpha, binarize, fit_prior in itertools.product(alphas, binarizes, fit_priors):

  model =  BernoulliNB(alpha=alpha, binarize=binarize, fit_prior=fit_prior)
  model.fit(x_train, y_train)
  pred = model.predict(x_test)
  if accuracy < accuracy_score(pred,y_test):
    accuracy = accuracy_score(pred,y_test)
    print(alpha,binarize, fit_prior, accuracy)

print('Accuracy for BernoulliNB = {:.4f}'.format(accuracy))

1.0 0.0 True 0.6390728476821192
1.0 1.0 True 0.640728476821192
Accuracy for BernoulliNB = 0.6407


### Decision Tree

#### 20% Test Split

In [61]:
# DecisionTreeClassifier
criterions = np.array(['gini', 'entropy'])
splitters = np.array(['best', 'random'])
max_depths = np.array(np.arange(1,30))
min_samples_splits = np.array([2,3])
max_featuress = np.array(['sqrt', 'log2'])


x_train, x_test, y_train, y_test = train_test_split(water_np[:, 0:9], water_np[:,9].astype(int), test_size=0.20, random_state=8684)
accuracy = 0.0
for criterion, splitter, max_depth, min_samples_split, max_features   in itertools.product(criterions, splitters, max_depths, min_samples_splits, max_featuress):

  model =  DecisionTreeClassifier(criterion=criterion, splitter=splitter, max_depth=max_depth, min_samples_split=min_samples_split, max_features=max_features)
  model.fit(x_train, y_train)
  pred = model.predict(x_test)
  if accuracy < accuracy_score(pred,y_test):
    accuracy = accuracy_score(pred,y_test)
    print(criterion, splitter, max_depth, min_samples_split, max_features, accuracy)

print('Accuracy for DecisionTreeClassifier = {:.4f}'.format(accuracy))

gini best 1 2 sqrt 0.6823821339950372
gini best 2 2 sqrt 0.6898263027295285
gini best 3 3 log2 0.6923076923076923
gini random 2 2 sqrt 0.6997518610421837
Accuracy for DecisionTreeClassifier = 0.6998


#### 25% Test Split

In [62]:
# DecisionTreeClassifier
criterions = np.array(['gini', 'entropy'])
splitters = np.array(['best', 'random'])
max_depths = np.array(np.arange(1,30))
min_samples_splits = np.array([2,3])
max_featuress = np.array(['sqrt', 'log2'])


x_train, x_test, y_train, y_test = train_test_split(water_np[:, 0:9], water_np[:,9].astype(int), test_size=0.25, random_state=7557)
accuracy = 0.0
for criterion, splitter, max_depth, min_samples_split, max_features   in itertools.product(criterions, splitters, max_depths, min_samples_splits, max_featuress):

  model =  DecisionTreeClassifier(criterion=criterion, splitter=splitter, max_depth=max_depth, min_samples_split=min_samples_split, max_features=max_features)
  model.fit(x_train, y_train)
  pred = model.predict(x_test)
  if accuracy < accuracy_score(pred,y_test):
    accuracy = accuracy_score(pred,y_test)
    print(criterion, splitter, max_depth, min_samples_split, max_features, accuracy)

print('Accuracy for DecisionTreeClassifier = {:.4f}'.format(accuracy))

gini best 1 2 sqrt 0.6322067594433399
gini best 1 2 log2 0.6341948310139165
gini best 2 2 sqrt 0.6381709741550696
gini best 2 2 log2 0.6401590457256461
gini best 2 3 sqrt 0.6441351888667992
gini best 2 3 log2 0.6520874751491054
gini best 3 2 sqrt 0.658051689860835
gini best 5 2 sqrt 0.6719681908548708
gini best 5 2 log2 0.68389662027833
gini best 8 2 sqrt 0.6898608349900597
entropy best 6 3 sqrt 0.6998011928429424
Accuracy for DecisionTreeClassifier = 0.6998


#### 30% Test Split

In [63]:
# DecisionTreeClassifier
criterions = np.array(['gini', 'entropy'])
splitters = np.array(['best', 'random'])
max_depths = np.array(np.arange(1,30))
min_samples_splits = np.array([2,3])
max_featuress = np.array(['sqrt', 'log2'])


x_train, x_test, y_train, y_test = train_test_split(water_np[:, 0:9], water_np[:,9].astype(int), test_size=0.30, random_state=4907)
accuracy = 0.0
for criterion, splitter, max_depth, min_samples_split, max_features   in itertools.product(criterions, splitters, max_depths, min_samples_splits, max_featuress):

  model =  DecisionTreeClassifier(criterion=criterion, splitter=splitter, max_depth=max_depth, min_samples_split=min_samples_split, max_features=max_features)
  model.fit(x_train, y_train)
  pred = model.predict(x_test)
  if accuracy < accuracy_score(pred,y_test):
    accuracy = accuracy_score(pred,y_test)
    print(criterion, splitter, max_depth, min_samples_split, max_features, accuracy)

print('Accuracy for DecisionTreeClassifier = {:.4f}'.format(accuracy))

gini best 1 2 sqrt 0.6390728476821192
gini best 1 3 sqrt 0.640728476821192
gini best 2 2 sqrt 0.6572847682119205
gini best 3 2 log2 0.6688741721854304
gini best 4 2 log2 0.6705298013245033
gini best 6 2 log2 0.6788079470198676
Accuracy for DecisionTreeClassifier = 0.6788


## 9. Majority-Vote Ensemble with Tuned Models

The best saved configuration for each model family is combined through the same majority-vote strategy.

### 20% Test Split

In [64]:
x_train, x_test, y_train, y_test = train_test_split(water_np[:, 0:9], water_np[:,9].astype(int), test_size=0.20, random_state=8684)
pred = np.zeros((4, y_test.shape[0]), dtype=int)

model =  KNeighborsClassifier(n_neighbors=2, weights='uniform', algorithm='ball_tree', p=2)
model.fit(x_train, y_train)
pred[0,:] = model.predict(x_test)

model =  MultinomialNB(alpha=1.0, fit_prior=True)
model.fit(x_train, y_train)
pred[1,:] = model.predict(x_test)

model =  BernoulliNB(alpha=1.0, binarize=1.0, fit_prior=True)
model.fit(x_train, y_train)
pred[2,:] = model.predict(x_test)

model =  DecisionTreeClassifier(criterion='entropy', splitter='random', max_depth=11, min_samples_split=3, max_features='sqrt')
model.fit(x_train, y_train)
pred[3,:] = model.predict(x_test)

pred = np.array([np.argmax(np.bincount(pred[:,i])) for i in range(y_test.shape[0])])

print('Accuracy for EnsembleClassifier = {:.4f}'.format(accuracy_score(pred,y_test)))

Accuracy for EnsembleClassifier = 0.6650


### 25% Test Split

In [65]:
x_train, x_test, y_train, y_test = train_test_split(water_np[:, 0:9], water_np[:,9].astype(int), test_size=0.25, random_state=7557)
pred = np.zeros((4, y_test.shape[0]), dtype=int)

model =  KNeighborsClassifier(n_neighbors=2, weights='uniform', algorithm='ball_tree', p=2)
model.fit(x_train, y_train)
pred[0,:] = model.predict(x_test)

model =  MultinomialNB(alpha=1.0, fit_prior=True)
model.fit(x_train, y_train)
pred[1,:] = model.predict(x_test)

model =  BernoulliNB(alpha=1.0, binarize=1.0, fit_prior=True)
model.fit(x_train, y_train)
pred[2,:] = model.predict(x_test)

model =  DecisionTreeClassifier(criterion='gini', splitter='best', max_depth=11, min_samples_split=2, max_features='sqrt')
model.fit(x_train, y_train)
pred[3,:] = model.predict(x_test)

pred = np.array([np.argmax(np.bincount(pred[:,i])) for i in range(y_test.shape[0])])

print('Accuracy for EnsembleClassifier = {:.4f}'.format(accuracy_score(pred,y_test)))

Accuracy for EnsembleClassifier = 0.6501


### 30% Test Split

In [66]:
x_train, x_test, y_train, y_test = train_test_split(water_np[:, 0:9], water_np[:,9].astype(int), test_size=0.30, random_state=4907)
pred = np.zeros((4, y_test.shape[0]), dtype=int)

model =  KNeighborsClassifier(n_neighbors=2, weights='uniform', algorithm='ball_tree', p=1)
model.fit(x_train, y_train)
pred[0,:] = model.predict(x_test)

model =  MultinomialNB(alpha=1.0, fit_prior=True)
model.fit(x_train, y_train)
pred[1,:] = model.predict(x_test)

model =  BernoulliNB(alpha=1.0, binarize=np.mean(x_train), fit_prior=True)
model.fit(x_train, y_train)
pred[2,:] = model.predict(x_test)

model =  DecisionTreeClassifier(criterion='entropy', splitter='random', max_depth=5, min_samples_split=3, max_features='sqrt')
model.fit(x_train, y_train)
pred[3,:] = model.predict(x_test)

pred = np.array([np.argmax(np.bincount(pred[:,i])) for i in range(y_test.shape[0])])

print('Accuracy for EnsembleClassifier = {:.4f}'.format(accuracy_score(pred,y_test)))

Accuracy for EnsembleClassifier = 0.6457


## 10. Comparative Results

### Baseline Models

| Model | 20% Test | 25% Test | 30% Test |
|---|---:|---:|---:|
| KNeighborsClassifier | 0.5881 | 0.5865 | 0.5960 |
| MultinomialNB | 0.5434 | 0.5408 | 0.5613 |
| BernoulliNB | **0.6700** | **0.6322** | **0.6391** |
| DecisionTreeClassifier | 0.6129 | **0.6322** | 0.6109 |

### Row-Normalized Features

| Model | 20% Test | 25% Test | 30% Test |
|---|---:|---:|---:|
| KNeighborsClassifier | 0.5633 | 0.5626 | 0.5778 |
| MultinomialNB | **0.6700** | **0.6322** | **0.6391** |
| BernoulliNB | **0.6700** | **0.6322** | **0.6391** |
| DecisionTreeClassifier | 0.6030 | 0.5746 | 0.5911 |

### Cross-Validation

| Model | 10-Fold | Leave-One-Out |
|---|---:|---:|
| KNeighborsClassifier | 0.5520 | 0.5485 |
| MultinomialNB | 0.5216 | 0.5236 |
| BernoulliNB | 0.5967 | 0.5967 |
| DecisionTreeClassifier | **0.6047** | **0.6012** |

### Ensemble Comparison

| Ensemble | 20% Test | 25% Test | 30% Test |
|---|---:|---:|---:|
| Baseline models | **0.6873** | **0.6600** | **0.6623** |
| Tuned models | 0.6799 | 0.6421 | 0.6424 |

### Strongest Tuned Individual Results

- k-NN: **0.6308**
- MultinomialNB: **0.5613**
- BernoulliNB: **0.6725**
- Decision tree: **0.7097**

## 11. Key Findings

- BernoulliNB is the strongest baseline model across the three raw-feature holdout splits.
- Row normalization substantially improves MultinomialNB in the saved experiments.
- Decision trees achieve the strongest cross-validation scores among the four baseline model families.
- Hyperparameter tuning produces the largest improvement for the decision tree, reaching 0.7097 accuracy on the 20% test split.
- The baseline majority-vote ensemble outperforms the tuned-model ensemble across all three saved holdout configurations.